# Learning 9: Memory & State Persistence

**Goal**: Learn how to persist state across conversations

## What You'll Learn
- Checkpointing and persistence
- Resuming conversations
- Different memory backends

In [1]:
from dotenv import load_dotenv
load_dotenv()

from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, AIMessage
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver
from typing import TypedDict, Annotated
import operator
from IPython.display import Image, display

llm = ChatOpenAI(model="gpt-4o-mini")
print("Setup complete!")

/Users/syedraza/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


Setup complete!


## Why Memory?

Without memory:
- Each conversation starts fresh
- No context from previous interactions
- Can't resume interrupted workflows

With memory:
- Remember user preferences
- Continue multi-turn conversations
- Resume workflows after crashes

## Creating a Checkpointer

A checkpointer saves state after each node execution.

In [2]:
# In-memory checkpointer (for development)
memory = MemorySaver()

print("Memory checkpointer created!")
print("Note: This is in-memory. For production, use SqliteSaver or PostgresSaver.")

Memory checkpointer created!
Note: This is in-memory. For production, use SqliteSaver or PostgresSaver.


## Building a Chat Agent with Memory

In [3]:
class ChatState(TypedDict):
    messages: Annotated[list, operator.add]

def chat_node(state: ChatState) -> dict:
    """Simple chat node that responds to messages."""
    response = llm.invoke(state["messages"])
    return {"messages": [response]}

# Build the graph
builder = StateGraph(ChatState)
builder.add_node("chat", chat_node)
builder.add_edge(START, "chat")
builder.add_edge("chat", END)

# Compile WITH checkpointer
chat_agent = builder.compile(checkpointer=memory)

print("Chat agent with memory compiled!")

Chat agent with memory compiled!


## Using Thread IDs

Thread IDs identify different conversations. Each thread has its own memory.

In [4]:
# Configuration with thread_id
config = {"configurable": {"thread_id": "user-123"}}

# First message
result1 = chat_agent.invoke(
    {"messages": [HumanMessage(content="Hi! My name is Alice.")]},
    config
)
print("Response 1:", result1["messages"][-1].content)

Response 1: Hi Alice! How can I assist you today?


In [5]:
# Second message - the agent remembers!
result2 = chat_agent.invoke(
    {"messages": [HumanMessage(content="What's my name?")]},
    config  # Same thread_id
)
print("Response 2:", result2["messages"][-1].content)

Response 2: Your name is Alice! How can I help you today?


In [6]:
# Different thread - doesn't know Alice
config2 = {"configurable": {"thread_id": "user-456"}}

result3 = chat_agent.invoke(
    {"messages": [HumanMessage(content="What's my name?")]},
    config2  # Different thread!
)
print("Response 3 (different thread):", result3["messages"][-1].content)

Response 3 (different thread): I'm sorry, but I don't know your name. If you'd like to share it or have any specific questions, feel free to let me know!


## Viewing State History

In [7]:
# Get the current state for a thread
state = chat_agent.get_state(config)

print("Current state for thread 'user-123':")
print(f"Number of messages: {len(state.values['messages'])}")
print("\nMessages:")
for i, msg in enumerate(state.values['messages']):
    print(f"  [{i}] {type(msg).__name__}: {msg.content[:50]}...")

Current state for thread 'user-123':
Number of messages: 4

Messages:
  [0] HumanMessage: Hi! My name is Alice....
  [1] AIMessage: Hi Alice! How can I assist you today?...
  [2] HumanMessage: What's my name?...
  [3] AIMessage: Your name is Alice! How can I help you today?...


In [ ]:
# View state history (all checkpoints)
print("State history:")
for state in chat_agent.get_state_history(config):
    print(f"  Checkpoint: {state.config['configurable']['checkpoint_id'][:20]}...")
    print(f"  Messages: {len(state.values['messages'])}")
    print()

## Example: User Preferences Memory

In [ ]:
class PreferenceState(TypedDict):
    messages: Annotated[list, operator.add]
    user_name: str
    favorite_color: str
    preferences_set: bool

def extract_preferences(state: PreferenceState) -> dict:
    """Extract user preferences from the conversation."""
    last_message = state["messages"][-1].content.lower()
    updates = {}
    
    # Simple extraction (in real apps, use LLM for this)
    if "my name is" in last_message:
        name = last_message.split("my name is")[-1].strip().split()[0].capitalize()
        updates["user_name"] = name
    
    if "favorite color is" in last_message:
        color = last_message.split("favorite color is")[-1].strip().split()[0]
        updates["favorite_color"] = color
    
    if updates:
        updates["preferences_set"] = True
    
    return updates

def respond_with_preferences(state: PreferenceState) -> dict:
    """Generate response using stored preferences."""
    # Include preferences in the system message
    system_context = "You are a helpful assistant."
    if state.get("user_name"):
        system_context += f" The user's name is {state['user_name']}."
    if state.get("favorite_color"):
        system_context += f" Their favorite color is {state['favorite_color']}."
    
    from langchain_core.messages import SystemMessage
    messages = [SystemMessage(content=system_context)] + state["messages"]
    
    response = llm.invoke(messages)
    return {"messages": [response]}

In [ ]:
# Build preference-aware agent
builder = StateGraph(PreferenceState)

builder.add_node("extract", extract_preferences)
builder.add_node("respond", respond_with_preferences)

builder.add_edge(START, "extract")
builder.add_edge("extract", "respond")
builder.add_edge("respond", END)

pref_memory = MemorySaver()
pref_agent = builder.compile(checkpointer=pref_memory)

In [ ]:
# Test the preference-aware agent
config = {"configurable": {"thread_id": "pref-user-1"}}

# Set preferences
result = pref_agent.invoke(
    {"messages": [HumanMessage(content="Hi! My name is Bob and my favorite color is blue.")],
     "user_name": "", "favorite_color": "", "preferences_set": False},
    config
)
print("Response:", result["messages"][-1].content)
print(f"\nStored - Name: {result['user_name']}, Color: {result['favorite_color']}")

In [ ]:
# Later conversation - remembers preferences!
result2 = pref_agent.invoke(
    {"messages": [HumanMessage(content="Can you recommend something in my favorite color?")]},
    config
)
print("Response:", result2["messages"][-1].content)

## Production Checkpointers

For production, use persistent storage:

In [ ]:
# SQLite (file-based, good for single-server)
# from langgraph.checkpoint.sqlite import SqliteSaver
# memory = SqliteSaver.from_conn_string(":memory:")  # or "./checkpoints.db"

# PostgreSQL (good for distributed systems)
# from langgraph.checkpoint.postgres import PostgresSaver
# memory = PostgresSaver.from_conn_string("postgresql://...")

print("Available checkpointers:")
print("  - MemorySaver: In-memory (development)")
print("  - SqliteSaver: SQLite database (single server)")
print("  - PostgresSaver: PostgreSQL (distributed)")

## Exercise: Build a Todo List Agent with Memory

Create an agent that:
1. Remembers a user's todo list
2. Can add/remove items
3. Persists across conversations

In [ ]:
# Your code here!



## Key Takeaways

1. Checkpointers save state after each node
2. Thread IDs separate different conversations
3. `get_state()` retrieves current state
4. `get_state_history()` shows all checkpoints
5. Use `MemorySaver` for dev, `SqliteSaver`/`PostgresSaver` for production

**Next**: Learning 10 - Human in the Loop